In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('/content/final_output_with_skills_sample.csv')
data.head(2)

,job_id,company_id,company_name,title,location,description_company,formatted_experience_level,work_type,Salary,name,...,state,country,city,zip_code,address,url,description,skills,experience_level,education
0,921716,2774458.0,Corcoran Sawyer Smith,Marketing Coordinator,"Princeton, NJ",Job descriptionA leading real estate firm in N...,Unknown,FULL_TIME,38480.0,Corcoran Sawyer Smith,...,NJ,US,Jersey City,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,With years of experience helping local buyers ...,"api development, government, marketing",NaN,NaN
1,10998357,64896719.0,The National Exemplar,Assitant Restaurant Manager,"Cincinnati, OH",The National Exemplar is accepting application...,Unknown,FULL_TIME,55000.0,The National Exemplar,...,Ohio,US,Mariemont,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,"In April of 1983, The National Exemplar began ...",NaN,NaN,NaN


In [ ]:
data.columns


Index(['job_id', 'company_id', 'company_name', 'title', 'location',
       'description_company', 'formatted_experience_level', 'work_type',
       'Salary', 'name', 'description_posting', 'company_size', 'state',
       'country', 'city', 'zip_code', 'address', 'url', 'description',
       'skills', 'experience_level', 'education'],
      dtype='object')

In [ ]:
data['Experience'] = data['formatted_experience_level']

In [ ]:
data = data.drop(['formatted_experience_level','description_company','description_posting'],axis=1)

In [ ]:
data = data.drop(columns=['experience_level'],axis=1)

In [ ]:
data = data.drop(columns=['education'],axis=1)

In [ ]:
data.columns

Index(['job_id', 'company_id', 'company_name', 'title', 'location',
       'work_type', 'Salary', 'name', 'company_size', 'state', 'country',
       'city', 'zip_code', 'address', 'url', 'description', 'skills',
       'Experience'],
      dtype='object')

In [ ]:
data["Experience"].value_counts()

,count
Experience,
Mid-Senior level,9800
Entry level,8781
Unknown,8050
Associate,1969
Director,821
Internship,339
Executive,240


In [ ]:
data.head()

,job_id,company_id,company_name,title,location,work_type,Salary,name,company_size,state,country,city,zip_code,address,url,description,skills,Experience
0,921716,2774458.0,Corcoran Sawyer Smith,Marketing Coordinator,"Princeton, NJ",FULL_TIME,38480.0,Corcoran Sawyer Smith,2.0,NJ,US,Jersey City,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,With years of experience helping local buyers ...,"api development, government, marketing",Unknown
1,10998357,64896719.0,The National Exemplar,Assitant Restaurant Manager,"Cincinnati, OH",FULL_TIME,55000.0,The National Exemplar,1.0,Ohio,US,Mariemont,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,"In April of 1983, The National Exemplar began ...",NaN,Unknown
2,23221523,766262.0,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,"New Hyde Park, NY",FULL_TIME,157500.0,"Abrams Fensterman, LLP",2.0,New York,US,Lake Success,11042,3 Dakota Drive,https://www.linkedin.com/company/abrams-fenste...,"Abrams Fensterman, LLP is a full-service law f...","administrative, api development, finance, lega...",Unknown
3,91700727,1481176.0,Downtown Raleigh Alliance,Economic Development and Planning Intern,"Raleigh, NC",INTERNSHIP,35360.0,Downtown Raleigh Alliance,1.0,North Carolina,US,Raleigh,27601,333 Fayetteville St,https://www.linkedin.com/company/downtownralei...,Mission of the Downtown Raleigh Alliance (DRA)...,NaN,Unknown
4,1218575,721189.0,Children's Nebraska,Respiratory Therapist,"Omaha, NE",FULL_TIME,81500.0,Children's Nebraska,5.0,NE,US,Omaha,68114,8200 Dodge Street,https://www.linkedin.com/company/childrensnebr...,"At Children’s Nebraska, our mission is to impr...","cerner, collaboration, education, healthcare, ...",Unknown


In [ ]:
from google.colab import userdata
import os


os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import os
import pickle
import logging
import time
import numpy as np
import pandas as pd
import scipy.sparse
import torch
import re
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# ── Configuration ─────────────────────────────────────────────
MODEL_DIR         = "models"
DATA_PATH         = "/content/final_output_with_skills_sample.csv"
CITIES_PATH       = "/content/uscities.csv"
WORLD_CITIES_PATH = "/content/worldcities.csv"
CACHE_FILE        = os.path.join(MODEL_DIR, "geo_cache.pkl")
BATCH_SIZE        = 512

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)


# 1. DATA

def load_and_clean(path):
    logger.info(f"Loading data from {path}")
    df = pd.read_csv(path).fillna("")

    # Rename experience column
    if "formatted_experience_level" in df.columns and "Experience" not in df.columns:
        df["Experience"] = df["formatted_experience_level"]

    # Drop unused columns safely
    for col in ["formatted_experience_level", "description_company",
                "description_posting", "experience_level", "education"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

    # Salary — convert to float
    def parse_salary(val):
        try:
            return float(str(val).replace(",", "").replace("$", "").strip())
        except:
            return np.nan

    df["salary_num"] = df["Salary"].apply(parse_salary)

    # Experience rank
    EXP_RANK = {
        "internship": 1, "entry level": 2, "associate": 3,
        "mid-senior level": 4, "director": 5, "executive": 6,
        "unknown": 0, "": 0
    }
    df["exp_rank"] = (
        df["Experience"].str.lower().str.strip()
        .map(EXP_RANK).fillna(0)
    )

    # Combined text for semantic search
    # Use 'description' column if it exists, otherwise empty string
    desc_col = "description" if "description" in df.columns else None
    desc_text = df[desc_col].str.lower().str[:300].fillna("") if desc_col else ""

    df["combined_text"] = (
        df["title"].str.lower().fillna("") + " " +
        df["skills"].str.lower().fillna("") + " " +
        desc_text
    ).str.strip()

    logger.info(f"Data loaded: {len(df)} rows, {len(df.columns)} columns")
    logger.info(f"Columns: {df.columns.tolist()}")
    return df

# 2. GEOCODING

def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "rb") as f:
            cache = pickle.load(f)
        logger.info(f"Loaded existing geo cache: {len(cache)} entries")
        return cache
    logger.info("No existing geo cache — starting fresh")
    return {}

def save_cache(cache):
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(cache, f)
    logger.info(f"Geo cache saved: {len(cache)} entries")

def build_city_lookup(us_cities_path, world_cities_path):
    logger.info("Building offline city lookup (US + world)...")
    lookup = {}

    # World cities first — base layer (lower priority)
    if os.path.exists(world_cities_path):
        world_df = pd.read_csv(world_cities_path)
        for _, row in world_df.iterrows():
            try:
                city    = str(row["city"]).strip().lower()
                country = str(row.get("country", "")).strip().lower()
                coords  = (float(row["lat"]), float(row["lng"]))

                if country:
                    key = f"{city}, {country}"
                    if key not in lookup:
                        lookup[key] = coords
                if city not in lookup:
                    lookup[city] = coords
            except (ValueError, KeyError):
                continue
        logger.info(f"World cities loaded: {len(lookup)} entries")
    else:
        logger.warning(
            f"worldcities.csv not found at {world_cities_path} — "
            "Download free: https://simplemaps.com/data/world-cities"
        )

    # US cities on top — higher priority, more precise
    us_count_before = len(lookup)
    if os.path.exists(us_cities_path):
        us_df = pd.read_csv(us_cities_path)
        for _, row in us_df.iterrows():
            try:
                city       = str(row["city"]).strip().lower()
                state_id   = str(row["state_id"]).strip().lower()
                state_name = str(row["state_name"]).strip().lower()
                coords     = (float(row["lat"]), float(row["lng"]))

                lookup[f"{city}, {state_id}"]    = coords   # "new york, ny"
                lookup[f"{city}, {state_name}"]  = coords   # "new york, new york"
                lookup[f"{city}, united states"] = coords   # "new york, united states"
                lookup[f"{city}, usa"]           = coords   # "new york, usa"
                lookup[f"{city}, us"]            = coords   # "new york, us"
                lookup[city]                     = coords   # overwrite world entry

            except (ValueError, KeyError):
                continue
        logger.info(f"US cities added: {len(lookup) - us_count_before} new entries")
    else:
        logger.warning(
            f"uscities.csv not found at {us_cities_path} — "
            "Download free: https://simplemaps.com/data/us-cities"
        )

    logger.info(f"Total city lookup: {len(lookup)} entries")
    return lookup

def detect_country_hint(location_str):
    loc = str(location_str).lower()

    # Order matters — more specific patterns first
    # Fix: ", in" conflict resolved by checking Indiana before India
    country_hints = [
        # US state abbreviations (check before India's "in")
        (", ny", "us"), (", ca", "us"), (", tx", "us"), (", fl", "us"),
        (", il", "us"), (", pa", "us"), (", oh", "us"), (", ga", "us"),
        (", nc", "us"), (", mi", "us"), (", wa", "us"), (", az", "us"),
        (", ma", "us"), (", tn", "us"), (", co", "us"), (", va", "us"),
        (", nj", "us"), (", mo", "us"), (", wi", "us"), (", mn", "us"),
        (", md", "us"), (", or", "us"), (", sc", "us"), (", in", "us"),
        ("united states", "us"), (", usa", "us"), (", u.s.", "us"),

        # Canada (check before India — both have "in")
        (", on", "canada"), (", bc", "canada"), (", ab", "canada"),
        (", qc", "canada"), (", mb", "canada"), (", sk", "canada"),
        ("canada", "canada"), (", ontario", "canada"),

        # UK
        ("united kingdom", "united kingdom"), (", uk", "united kingdom"),
        (", england", "united kingdom"), ("great britain", "united kingdom"),

        # India — checked after US/Canada to avoid ", in" conflict
        ("india", "india"), (", india", "india"),

        # Australia
        ("australia", "australia"), (", au", "australia"),
        (", nsw", "australia"), (", vic", "australia"), (", qld", "australia"),

        # Germany
        ("germany", "germany"), (", de", "germany"),

        # Others
        ("france", "france"), ("spain", "spain"), ("italy", "italy"),
        ("netherlands", "netherlands"), ("singapore", "singapore"),
        ("japan", "japan"), ("china", "china"), ("brazil", "brazil"),
        ("mexico", "mexico"), ("ireland", "ireland"),
    ]

    # Use a list of tuples so order is preserved
    for indicator, country in country_hints:
        if indicator in loc:
            return country

    return None

def parse_location(location_str):
    if not location_str or str(location_str).strip() == "":
        return None, None, None

    loc = str(location_str).strip().lower()

    # Skip non-geographic entries
    if any(x in loc for x in [
        "remote", "anywhere", "nationwide", "work from home",
        "hybrid", "telecommute", "virtual", "no office"
    ]):
        return None, None, "remote"

    # Remove zip codes
    loc = re.sub(r'\b\d{5}(-\d{4})?\b', '', loc)           # US zip
    loc = re.sub(r'\b[a-z]\d[a-z]\s?\d[a-z]\d\b', '', loc) # Canada postal
    loc = re.sub(r'\s+', ' ', loc).strip()

    country_hint = detect_country_hint(location_str)
    parts        = [p.strip() for p in loc.split(",") if p.strip()]

    if not parts:
        return None, None, country_hint
    if len(parts) == 1:
        return parts[0], None, country_hint

    return parts[0], parts[1], country_hint

def get_coords_offline(location_str, city_lookup, cache):
    if not location_str:
        return None

    key = str(location_str).strip().lower()

    if key in cache:
        return cache[key]

    city, region, country_hint = parse_location(location_str)

    if country_hint == "remote":
        cache[key] = None
        return None

    if city is None:
        cache[key] = None
        return None

    result = None

    # Attempt 1 — city + region (most precise)
    if region:
        result = city_lookup.get(f"{city}, {region}")

    # Attempt 2 — city + country hint
    if result is None and country_hint:
        result = city_lookup.get(f"{city}, {country_hint}")

    # Attempt 3 — city only
    if result is None:
        result = city_lookup.get(city)

    # Attempt 4 — strip common geographic prefixes
    if result is None:
        for prefix in ["greater ", "metro ", "downtown ",
                       "north ", "south ", "east ", "west ", "central "]:
            if city.startswith(prefix):
                trimmed   = city[len(prefix):]
                candidate = f"{trimmed}, {region}" if region else trimmed
                result    = city_lookup.get(candidate) or city_lookup.get(trimmed)
                if result:
                    break

    cache[key] = result
    return result

def geocode_all(df, us_cities_path, world_cities_path, cache_file):
    city_lookup = build_city_lookup(us_cities_path, world_cities_path)
    cache       = load_cache()

    unique_locs = df["location"].dropna().unique()
    logger.info(f"Geocoding {len(unique_locs)} unique locations...")

    new_count = 0
    for loc in tqdm(unique_locs, desc="Geocoding"):
        key = str(loc).strip().lower()
        if key not in cache:
            get_coords_offline(loc, city_lookup, cache)
            new_count += 1

    save_cache(cache)

    df["coords"] = df["location"].apply(
        lambda x: cache.get(str(x).strip().lower())
    )

    covered = df["coords"].notna().sum()
    remote  = df["location"].str.lower().str.contains(
        "remote|anywhere|virtual|work from home", na=False
    ).sum()
    total   = len(df)

    logger.info(f"Coverage:     {covered}/{total} rows ({100*covered//total}%)")
    logger.info(f"Remote jobs:  {remote} (no coords expected)")
    logger.info(f"New lookups:  {new_count}")

    return df, cache


# 3. FEATURE
def build_features(df):
    logger.info("Building TF-IDF matrix...")
    tfidf  = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2)
    matrix = tfidf.fit_transform(df["skills"].fillna(""))
    logger.info(f"TF-IDF shape: {matrix.shape}")

    logger.info("Building semantic embeddings...")
    device     = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Device: {device}")

    model      = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    batch_size = 1024 if device == "cuda" else BATCH_SIZE

    embeddings = model.encode(
        df["combined_text"].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        device=device,
        convert_to_numpy=True,
    )
    logger.info(f"Embeddings shape: {embeddings.shape}")

    return tfidf, matrix, embeddings


# 4.

def save_and_validate(df, tfidf, matrix, embeddings):
    logger.info("Saving models...")
    os.makedirs(MODEL_DIR, exist_ok=True)

    paths = {
        "tfidf_vectorizer": os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"),
        "tfidf_matrix":     os.path.join(MODEL_DIR, "tfidf_matrix.npz"),
        "job_embeddings":   os.path.join(MODEL_DIR, "job_embeddings.npy"),
        "jobs_clean":       os.path.join(MODEL_DIR, "jobs_clean.pkl"),
    }

    with open(paths["tfidf_vectorizer"], "wb") as f:
        pickle.dump(tfidf, f)

    scipy.sparse.save_npz(paths["tfidf_matrix"], matrix)
    np.save(paths["job_embeddings"], embeddings)
    df.to_pickle(paths["jobs_clean"])

    logger.info("Validating saved files...")
    all_ok = True
    for name, path in paths.items():
        exists = os.path.exists(path)
        size   = os.path.getsize(path) if exists else 0
        status = "OK" if exists and size > 0 else "FAILED"
        logger.info(f"  {status}  {name}: {size:,} bytes")
        if status == "FAILED":
            all_ok = False

    if not all_ok:
        raise RuntimeError("Model saving validation failed — check logs above")

    logger.info("All files saved and validated")


# 5.

def print_summary(df, matrix, embeddings, elapsed):
    logger.info("=" * 50)
    logger.info("PIPELINE SUMMARY")
    logger.info("=" * 50)
    logger.info(f"Total rows:          {len(df)}")
    logger.info(f"Columns:             {len(df.columns)}")
    logger.info(f"TF-IDF shape:        {matrix.shape}")
    logger.info(f"Embeddings shape:    {embeddings.shape}")
    logger.info(f"Geocoded rows:       {df['coords'].notna().sum()}")
    logger.info(f"Salary available:    {df['salary_num'].notna().sum()}")
    logger.info(f"Experience labeled:  {(df['exp_rank'] > 0).sum()}")
    logger.info(f"Time taken:          {elapsed:.1f}s")
    logger.info("=" * 50)
    logger.info("Next step: uvicorn api:app --reload --port 8000")


if __name__ == "__main__":
    start = time.time()
    os.makedirs(MODEL_DIR, exist_ok=True)

    df                        = load_and_clean(DATA_PATH)
    df, geo_cache             = geocode_all(df, CITIES_PATH, WORLD_CITIES_PATH, CACHE_FILE)
    tfidf, matrix, embeddings = build_features(df)
    save_and_validate(df, tfidf, matrix, embeddings)

    elapsed = time.time() - start
    print_summary(df, matrix, embeddings, elapsed)

Geocoding: 100%|██████████| 4585/4585 [00:00<00:00, 71570.36it/s]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

source_folder = 'models'
destination_folder = '/content/drive/MyDrive/models_backup'

# Create the destination folder in Google Drive if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Copy the contents of the models folder to Google Drive
shutil.copytree(source_folder, destination_folder, dirs_exist_ok=True)

print(f"Models folder saved to: {destination_folder}")

Models folder saved to: /content/drive/MyDrive/models_backup
